# Session 6 lab: SVD and PCA

**Mathematics for AI and Machine Learning, KLAi Professional Learning.**

You will find the singular values of a small matrix, remove noise from an 8 by 8 image with a low-rank approximation, run principal component analysis (PCA) on the eight marks from Session 5, find the singular values of the image again by power iteration and deflation, and count the numbers that LoRA saves. Every step has the rhythm PREDICT, run, READ, CHANGE one thing. After each answer a check line says OK or NOT YET. The image and the marks are illustrative (made-up numbers to show the arithmetic).

Run the cells from the top, in order. A cell that starts with a comment and the words `your code here` is yours to complete.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("numpy is ready, version", np.__version__)

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks(total=None):
    # total = how many checks this notebook has; a skipped or crashed cell still counts as NOT YET
    k = sum(_results.values()); n = max(total or 0, len(_results))
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

---
## Step 1: The SVD of a small matrix

The recipe slide shows the NumPy function for the SVD. It returns three things: U, the singular values S as a list (largest first), and V transpose, written Vt. A equals U times the diagonal matrix of S times Vt. The singular values are also the square roots of the eigenvalues of AᵀA.

**PREDICT first.** A has rows (2, 2) and (−1, 2). By hand, AᵀA has rows (5, 2) and (2, 8), with trace 13 and determinant 36. What are its eigenvalues, and so the singular values of A?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
A = np.array([[2.0, 2.0], [-1.0, 2.0]])
print("A =\n", A)

In [ ]:
# Get U, S and Vt (recipe slide). Then the eigenvalues of AᵀA (largest first, use the symmetric eigenvalue function from Session 5) and the matrix rebuilt from the three parts.
U = None  # your code here
S = None  # your code here
Vt = None  # your code here
lam_ata = None  # your code here
rebuilt = None  # your code here

In [ ]:
print("singular values:", S)
print("eigenvalues of A^T A:", lam_ata)
check("singular values are 3 and 2", S, [3.0, 2.0])
check("square roots of the eigenvalues of A^T A agree", None if lam_ata is None else np.sqrt(lam_ata), [3.0, 2.0])
check("U, S and Vt rebuild A", rebuilt, A)

**READ the output.** Are the singular values listed from largest to smallest? Print Vt: its rows are the right singular vectors. What is the length of each row?

> My answer: 

**CHANGE one thing.** Run the next cell: a matrix whose second row is twice the first. Predict its second singular value first.

> What I saw: 

In [ ]:
S_flat = np.linalg.svd(np.array([[1.0, 2.0], [2.0, 4.0]]))[1]
print("singular values:", np.round(S_flat, 6), "so the rank is", int(np.sum(S_flat > 1e-9)))

---
## Step 2: An 8 by 8 image with noise

A clean H pattern (cells 1 or 8, on a scale from 0 white to 9 black) plus random noise of 0, 1 or 2 up or down. We take the SVD of the noisy image and look at its singular values.

In [ ]:
clean = np.ones((8, 8))
clean[:, [1, 6]] = 8
clean[3:5, 1:7] = 8
noise = np.random.default_rng(39).integers(-2, 3, size=(8, 8))
noisy = np.clip(clean + noise, 0, 9)
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(clean, cmap="Greys", vmin=0, vmax=9)
ax[0].set_title("clean")
ax[1].imshow(noisy, cmap="Greys", vmin=0, vmax=9)
ax[1].set_title("noisy")
plt.show()
print("error of the noisy image from the clean one:", round(np.linalg.norm(noisy - clean), 2))

**PREDICT first.** How many singular values of the noisy image will be large? The clean H has two different kinds of rows, so how many independent patterns does it hold?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# SVD of the noisy image. Then the share of each squared singular value, its running total (cumulative share), and the smallest k that reaches 95 percent.
U2 = None  # your code here
S2 = None  # your code here
Vt2 = None  # your code here
cum2 = None  # your code here
k95 = None  # your code here

In [ ]:
print("singular values:", None if S2 is None else np.round(S2, 2))
print("cumulative share:", None if cum2 is None else np.round(cum2, 3))
print("k for 95 percent:", k95)
check("first two singular values", None if S2 is None else S2[:2], [36.84568001427707, 15.070165507206665], 0.01)
check("two pieces hold 97 percent", None if cum2 is None else cum2[1], 0.9716211061490504, 0.001)
check("k for 95 percent is 2", k95, 2, 0)

**READ the output.** Why is there a gap after the second singular value? What does the flat tail of small values stand for?

> My answer: 

---
## Step 3: Low-rank approximations and their errors

The rank k approximation keeps the first k terms of the sum on the slide. The recipe slide shows how to slice U, S and Vt. We measure the error against the clean image, which we know only because we built it. The error is the square root of the sum of the squared cell differences.

**PREDICT first.** Which rank will give the smallest error from the clean image: 1, 2 or 3? Will rank 3 be better or worse than rank 2?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Write a function approx(k) that returns the rank k approximation of the noisy image (recipe slide). Then compute the three errors against the clean image, as defined on the image slide.
approx = None  # your code here
errs = None  # your code here

In [ ]:
print("errors for rank 1, 2, 3:", None if errs is None else np.round(errs, 2))
check("rank 1 error", None if errs is None else errs[0], 15.40072089984082, 0.01)
check("rank 2 error is the smallest", None if errs is None else errs[1], 4.539092348703314, 0.01)
check("rank 3 error", None if errs is None else errs[2], 6.728651337181748, 0.01)

In [ ]:
if approx is not None:
    fig, ax = plt.subplots(1, 4, figsize=(9, 2.5))
    for a, (img, t) in zip(ax, [(noisy, "noisy"), (approx(1), "rank 1"), (approx(2), "rank 2"), (approx(3), "rank 3")]):
        a.imshow(img, cmap="Greys", vmin=0, vmax=9)
        a.set_title(t)
        a.axis("off")
    plt.show()

**READ the output.** Does rank 2 look more like the clean H than the noisy image does? Why does the error rise again at rank 3?

> My answer: 

**CHANGE one thing.** Run the next cell: the rank 4 error. Predict whether it is higher or lower than rank 3. How many numbers does rank 4 store, 4 × (8 + 8 + 1), compared with 64?

> What I saw: 

In [ ]:
if approx is not None:
    print("rank 4 error:", round(np.linalg.norm(approx(4) - clean), 2), " numbers stored:", 4 * 17, "against 64")

---
## Step 4: PCA of the eight marks, step by step

The same eight students as Session 5. Centre the columns, build the covariance matrix, take the top eigenvector, give each student a score (the dot product with it), and measure the explained variance.

In [ ]:
maths = np.array([53.0, 55.0, 63.0, 67.0, 48.0, 67.0, 56.0, 71.0])
phys = np.array([53.0, 52.0, 62.0, 65.0, 49.0, 61.0, 57.0, 65.0])
X = np.column_stack([maths, phys])
print("means:", X.mean(axis=0))

**PREDICT first.** Last session the eigenvalues were 102 and 2 and the top direction (0.8, 0.6). What will the variance of the scores along that direction be? What share of the total variance is that?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Centre X, build C, get the top eigenvector (largest eigenvalue), the score of each student (the dot product of the centred marks with the top eigenvector), the variance of the scores (divide by n − 1), the explained variance ratio and the squared distance lost.
Xc = None  # your code here
C = None  # your code here
top = None  # your code here
scores = None  # your code here
var_scores = None  # your code here
ratio = None  # your code here
lost = None  # your code here

In [ ]:
print("top component:", top)
print("scores:", None if scores is None else np.round(scores, 1))
print("variance of scores:", var_scores, " explained ratio:", ratio, " squared distance lost:", lost)
check("top component, up to a sign", None if top is None else np.abs(top), [0.8, 0.6], 0.001)
check("score of the first student", None if scores is None else abs(scores[0]), 8.6, 0.001)
check("variance of the scores is the top eigenvalue 102", var_scores, 102.0, 0.001)
check("explained variance ratio", ratio, 0.9807692307692307, 0.001)
check("squared distance lost is 7 × 2 = 14", lost, 14.0, 0.001)

**READ the output.** The second eigenvalue is 2. How is it linked to the squared distance lost, 14? What would happen to the ratio if we kept both components?

> My answer: 

### Step 4, continued: PCA is the SVD of the centred data

Take the SVD of Xc. The rows of Vt are the principal components, and the variance along each is the squared singular value divided by n - 1 (the sample variance of Session 5).

**PREDICT first.** What will the variances S**2 / (n - 1) be? Will the first row of Vt equal your top component?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# SVD of Xc (the recipe slide shows the version for tall data), then the variances along each component.
U3 = None  # your code here
S3 = None  # your code here
Vt3 = None  # your code here
var_svd = None  # your code here

In [ ]:
print("singular values of Xc:", S3)
print("variances:", var_svd)
check("variances are 102 and 2", var_svd, [102.0, 1.999999999999993], 0.001)
check("first row of Vt, up to a sign", None if Vt3 is None else np.abs(Vt3[0]), [0.8, 0.6], 0.001)

**CHANGE one thing.** Replace the physics marks by 100 minus physics, then run PCA again by SVD. Predict whether the explained variance ratio changes and which way the first component points.

> What I saw: 

In [ ]:
# Centre the new data and take its SVD. Compute the first component and the ratio.
top_flip = None  # your code here
ratio_flip = None  # your code here

In [ ]:
print("first component:", top_flip, " ratio:", ratio_flip)
check("explained ratio is unchanged", ratio_flip, 0.9807692307692307, 0.001)
check("component is (0.8, -0.6) up to a sign", None if top_flip is None else np.abs(top_flip), [0.8, 0.6], 0.001)
check("the signs of the two entries now differ", None if top_flip is None else float(np.sign(top_flip[0] * top_flip[1])), -1.0, 0)

---
## Step 5: Power iteration and deflation find the singular values

Neither Excel nor Google Sheets has an SVD, so the spreadsheet does this step with cells. The top eigenvector of M = noisyᵀ noisy is the first right singular vector v₁, and the top eigenvalue of M is σ₁². Then u₁ = noisy v₁ / σ₁ and the rank 1 term is σ₁ u₁ v₁ᵀ. (Lab slide for step 5: power iteration on NᵀN, then deflation, where N is the noisy image.)

In [ ]:
M = noisy.T @ noisy      # an 8 by 8 symmetric matrix
g = np.ones(8)            # the starting guess g0 = (1, 1, ..., 1)
print("M has shape", M.shape)

**PREDICT first.** After 10 multiplications by M, each followed by dividing by the largest entry, the largest entry should be σ₁². The slide gives σ₁ = 36.85, so what is its square?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Run 10 steps of power iteration on M, starting from g. Keep the last largest entry in est. Then sigma1, the unit vector v1, u1 and the rank 1 term.
est = None  # your code here
sigma1 = None  # your code here
v1 = None  # your code here
u1 = None  # your code here
rank1 = None  # your code here

In [ ]:
print("largest entry after 10 steps:", est, " sigma1:", sigma1)
check("largest entry is sigma1 squared, 1357.6", est, 1357.6041568062774, 0.1)
check("sigma1 agrees with the SVD", sigma1, 36.84568001427707, 0.001)
check("rank 1 term agrees with the SVD rank 1 approximation", None if (rank1 is None or approx is None) else np.max(np.abs(rank1 - approx(1))), 0, 0.001)

**READ the output.** The rank 1 term has 64 cells but is built from 8 + 8 + 1 numbers. Look at the cell in row 4, column 2 of rank1: in Python that is rank1[3, 1], because Python counts from 0. How does it compare with noisy[3, 1]? Values can leave the 0 to 9 range, because this is an approximation.

> My answer: 

**CHANGE one thing.** Deflation: subtract the rank 1 part of M (σ₁² times v₁ v₁ᵀ) so that its top eigenvalue is removed, then repeat the same 10 steps from a fresh start to find σ₂. Predict σ₂ first: the scree plot slide shows it.

> What I saw: 

In [ ]:
# Deflate M, repeat the power iteration from a fresh start, and keep est2, sigma2, v2, u2. Then add the second term to rank1 and measure the rank 2 error against the clean image.
est2 = None  # your code here
sigma2 = None  # your code here
rank2 = None  # your code here
err2 = None  # your code here

In [ ]:
print("largest entry of the second round:", est2, " sigma2:", sigma2)
print("rank 2 error from the clean image:", None if err2 is None else round(err2, 2))
check("sigma2 agrees with the SVD", sigma2, 15.070165507206665, 0.001)
check("the rank 2 error is 4.54, as in step 3", err2, 4.539092348703314, 0.01)

**READ the output.** Why does subtracting σ₁² v₁ v₁ᵀ leave the other eigenvectors of M unchanged? Predict σ₃ from the scree plot slide, then run the given third round below.

> My answer: 

In [ ]:
# A third round, given: deflate again, repeat the 10 steps, add the third term
if est2 is not None and rank2 is not None:
    M3 = M2 - est2 * np.outer(v2, v2)
    g = np.ones(8)
    for k in range(10):
        Mg = M3 @ g
        est3 = np.max(np.abs(Mg))
        g = Mg / est3
    sigma3 = np.sqrt(est3)
    v3 = g / np.linalg.norm(g)
    u3 = noisy @ v3 / sigma3
    rank3 = rank2 + sigma3 * np.outer(u3, v3)
    print("sigma3:", round(sigma3, 2), " rank 3 error from the clean image:", round(np.linalg.norm(rank3 - clean), 2))

---
## Step 6: LoRA: count the numbers

LoRA (Hu et al., 2021) keeps a pre-trained weight matrix W frozen and trains the change as B times A, with B of size d by r and A of size r by d. Here we only count numbers and check the rank, with illustrative sizes.

**PREDICT first.** For d = 1,000 and r = 8, how many numbers are in a full d by d update, and how many in B and A together? What fraction is that?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# Count the numbers in a full update and in B and A together. Then build a small B (50 by 3) and A (3 by 50) with random numbers and find the rank of their product.
full_update = None  # your code here
lora_count = None  # your code here
share = None  # your code here
rank_BA = None  # your code here

In [ ]:
print("full update:", full_update, " B and A:", lora_count, " share:", share, " rank of B A:", rank_BA)
check("full update numbers", full_update, 1000000, 0)
check("LoRA numbers", lora_count, 16000, 0)
check("share is 1.6 percent", share, 0.016, 0.0001)
check("B A is a 50 by 50 matrix of rank 3", rank_BA, 3, 0)

**READ the output.** A 50 by 50 matrix has 2,500 numbers, yet B A has rank 3. Why can such a product never have a rank above r?

> My answer: 

---
## Result

Run the next cell. It must say **All checks pass** before you save.

In [ ]:
show_checks(28)